# 04 — Reranking and Evidence Selection

## 1. Title and objective

Phase 3 improved recall by combining dense and BM25 rankings with Reciprocal Rank Fusion (RRF). Phase 4 improves precision after that retrieval boundary. It applies a local cross-encoder to the fused candidate pool, selects the smallest strong and sufficiently diverse evidence set, builds a token-bounded context, and generates a grounded answer with citations and evidence-quality diagnostics.

**Phase 4 status:** implemented and automated-test ready; full frozen-benchmark qualification is still pending. This notebook does not claim an answer-quality improvement until the gated benchmark is run with the approved local corpus and models.

### 1.1 Local-first experiment contract

- No cloud inference, cloud reranking API, or cloud vector database is used.
- The configured reranker must already exist in the local model cache or at a local path.
- Operational paths, model names, budgets, thresholds, and artifact names come from `Phase4Config`.
- Notebooks 01, 02, and 03 remain unchanged comparison baselines.
- Every executed manual, smoke, or benchmark run is exported under the configured Phase 4 run root.

## 2. Recap of Phase 3

Phase 3 retrieves each query variant through dense and BM25 retrievers. Dense similarity is useful for paraphrases and conceptual matches; BM25 is useful for identifiers, acronyms, titles, and exact policy language. RRF combines their rank positions without treating their raw scores as if they had the same units. The resulting hybrid list is a recall-oriented candidate pool, followed by deduplication, optional neighbor expansion, overlap merging, token-aware context construction, local generation, citations, and run artifacts.

## 3. Phase 3 limitations

High recall is not the same as high precision. An RRF candidate can rank well because one retriever found it early, while still being only weakly related to the complete question. Multi-query retrieval can also surface overlapping chunks, repeated sections from one document, and evidence that consumes tokens without changing the answer. Passing all available candidates to generation can increase latency, dilute strong evidence, and make citations harder to review.

Phase 4 therefore inserts an explicit decision boundary: **retrieve broadly, then justify every chunk retained for generation**.

## 4. Theory of reranking

A first-stage retriever is optimized to search a large corpus efficiently. A reranker works on the much smaller candidate set and can spend more computation evaluating each question/chunk pair. It does not replace hybrid retrieval; it consumes its output. This two-stage design separates recall from precision and makes the cost of the more expensive relevance model bounded by the candidate-pool size.

## 5. Cross-encoder explanation

A bi-encoder embeds the question and chunk independently, which makes corpus-scale search practical. A cross-encoder reads the question and candidate text together. Joint attention can evaluate relationships that independent embeddings miss, but it must run once per pair and is therefore appropriate only after retrieval.

`CrossEncoderReranker` loads lazily, supports configured CPU/GPU execution and batching, and records latency. Every process checks the local Hugging Face cache first. Developer mode (`reranker_local_files_only=False`) downloads and caches a missing model once; enterprise offline mode (`True`) skips download and fails with staging guidance. `MockReranker` implements the same interface for deterministic tests without model files or hardware assumptions.

### 5.1 Why reranking follows RRF

Dense cosine similarity, BM25 relevance, RRF score, and cross-encoder score are different signals with different scales. Directly averaging them would create an undocumented calibration assumption. RRF first combines only rank positions. The reranker then produces a separate question/chunk relevance ordering. All raw values remain in the trace for diagnosis; none are silently averaged.

## 6. Evidence selection theory

Reranking orders candidates; selection decides whether a candidate deserves context space. Phase 4 can combine five strategies:

1. `top_k` limits the evidence count.
2. `reranker_score_threshold` rejects weak candidates.
3. `source_diversity` caps concentration from one document.
4. `redundancy_reduction` rejects near-duplicate wording.
5. `token_budget` rejects evidence that would exceed the smaller evidence budget.

The selector does not try to fill the model window, but it also does not maximize token reduction blindly. It preserves a minimum evidence floor, targets roughly 800--1500 selected-evidence tokens for normal QA, and uses top-ranked fallback evidence when thresholding would otherwise retain nothing. Fallback chunks are marked weak/low-confidence rather than treated as no evidence. Every discarded chunk receives one normalized primary reason.

## 7. Phase 4 architecture

```text
Question
  → Phase 3 query variants + Dense/BM25 retrieval
  → Reciprocal Rank Fusion candidate pool
  → Local cross-encoder reranker
  → Evidence selector (score, diversity, redundancy, tokens, top-k)
  → Existing token-aware context builder
  → Existing grounded local LLM + citations
  → Evidence quality + Phase 4 trace
  → CSV / XLSX / standalone HTML / JSON / logs / context
```

The compatibility seam is the hand-off to the existing context builder. Phase 4 replaces the internal candidate list with selected evidence but retains Phase 2 and Phase 3 response keys, citation behavior, token management, and `RunManager` paths.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from cial_knowledge_os import (
    Phase3Config,
    Phase3RAGPipeline,
    Phase3Runner,
    Phase4Config,
    Phase4RAGPipeline,
    Phase4Runner,
    load_benchmark,
)

pd.set_option("display.max_colwidth", 120)
print(f"Project root: {PROJECT_ROOT}")
print("All retrieval, reranking, tokenization, and generation remain local.")

## 8. Configuration

The default evidence budget is intentionally smaller than the final context budget. This makes selection an optimization stage instead of a second name for truncation. The default floor is three chunks, the ceiling is eight, and normal QA targets 800--1500 selected-evidence tokens. The default `-4.0` threshold is appropriate for the configured MS MARCO cross-encoder's logit output; it is model-specific and remains configurable. Neighbor expansion defaults to off in Phase 4 so chunks that never passed reranking are not silently introduced after selection.

**Phase 3 versus Phase 4 answer style:** Phase 4 preserves Phase 3's strict grounding and citation contract while requesting a detailed, decision-useful synthesis. With `adaptive_answer_sections=True`, it chooses only question-relevant section families; set the option to `False` to restore the previous fixed template for reproducibility. Evidence selection reduces irrelevant context, not answer depth. Every implication, action, risk/gap, caveat, and decision note must come only from selected evidence; the word target never permits padding or unsupported claims. This is semi-adaptive section selection, not a full response planner; full adaptive response planning remains deferred to Phase 5, and no benchmark quality improvement is claimed from this prompt refinement.

In [ ]:
config = Phase4Config(
    project_root=PROJECT_ROOT,
    retrieval_mode="hybrid",
    dense_top_k=10,
    bm25_top_k=10,
    retrieval_top_k=10,
    rrf_k=60,
    max_context_tokens=4096,
    reranker_candidate_top_k=30,
    reranker_batch_size=16,
    reranker_local_files_only=False,  # cache first; download once if missing
    min_selected_evidence=3,
    max_selected_evidence=8,
    reranker_score_threshold=-4.0,
    fallback_to_top_n_if_empty=True,
    fallback_top_n=3,
    weak_evidence_answer_allowed=True,
    answer_detail_level="detailed",
    min_answer_words=250,
    max_answer_words=None,
    prefer_structured_answers=True,
    adaptive_answer_sections=True,
    include_decision_notes=True,
    generation_retries=2,
    retry_cooldown_seconds=20,
    evidence_token_budget=2400,
    selected_evidence_target_min_tokens=800,
    selected_evidence_target_max_tokens=1500,
    evidence_max_chunks_per_source=2,
    evidence_redundancy_threshold=0.85,
    phase4_trace_mode="full",
)

display(pd.DataFrame([
    ("reranker_model", config.reranker_model_name),
    ("reranker_device", config.reranker_device),
    ("local_files_only", config.reranker_local_files_only),
    ("candidate_top_k", config.reranker_candidate_top_k),
    ("min_selected_evidence", config.min_selected_evidence),
    ("max_selected_evidence", config.max_selected_evidence),
    ("reranker_score_threshold", config.reranker_score_threshold),
    ("fallback_top_n", config.fallback_top_n),
    ("weak_evidence_answer_allowed", config.weak_evidence_answer_allowed),
    ("answer_detail_level", config.answer_detail_level),
    ("min_answer_words", config.min_answer_words),
    ("max_answer_words", config.max_answer_words),
    ("prefer_structured_answers", config.prefer_structured_answers),
    ("adaptive_answer_sections", config.adaptive_answer_sections),
    ("include_decision_notes", config.include_decision_notes),
    ("generation_retries", config.generation_retries),
    ("retry_cooldown_seconds", config.retry_cooldown_seconds),
    ("evidence_token_budget", config.evidence_token_budget),
    ("selected_token_target", f"{config.selected_evidence_target_min_tokens}-{config.selected_evidence_target_max_tokens}"),
    ("final_context_budget", config.max_context_tokens),
    ("selection_strategies", ", ".join(config.evidence_selection_strategies)),
    ("output_root", config.output_root),
    ("phase_output_name", config.phase_output_name),
], columns=["setting", "effective_value"]))

### 8.1 Execution modes and trace depth

`Phase4Runner` supports `smoke`, `manual_qa`, `benchmark`, and `export_only`. Full traces retain candidate text for deep inspection. Compact traces retain ranks, metadata, previews, decisions, and diagnostics while avoiding very large JSON files. Manual QA runs are unlimited by default; set `max_inline_manual_questions` to a positive integer for a deliberate notebook smoke cap. For large or long-running batches, edit the `USER CONFIGURATION` section in `scripts/run_phase4_batch.py` and run that file directly (including VS Code **Run Python File**). No CLI arguments are required. The script preserves the notebook-equivalent exports and visualizations without tying execution to the Jupyter kernel. It checkpoints every question; set `RESUME_RUN_FOLDER`, retry/cooldown values, and `MAX_ANSWER_WORDS` in that section as needed. Optional CLI overrides remain available for advanced one-off runs; the notebook safety workflow remains unchanged.

The first answer prints one unambiguous model-loading status: loaded from the local cache, downloaded and cached successfully, or download skipped because enterprise offline mode is enabled. The response and Phase 4 trace also record `reranker_load_source` as `cache`, `download`, `injected`, or `mock`.

In [ ]:
RUN_LOCAL_PIPELINE = True  # Set True only after local models and Ollama are ready.
pipeline = Phase4RAGPipeline(config)

if RUN_LOCAL_PIPELINE:
    documents = pipeline.load()
    chunks = pipeline.chunk()
    vectors = pipeline.embed()
    pipeline.index()
    print({"documents": len(documents), "chunks": len(chunks), "vectors": len(vectors)})
else:
    print("Pipeline construction verified; corpus/model loading is gated.")

## 9. Single-query walkthrough

A deep-dive run should inspect retrieval before trusting generation. The response keeps Phase 3 keys and adds `candidate_pool`, `reranked_candidates`, `selected_evidence`, `discarded_evidence`, `evidence_quality`, `token_efficiency`, and a Phase 4 `question_trace`.

In [ ]:
DEEP_DIVE_QUESTION = "What controls should protect an enterprise AI assistant from prompt injection?"
response = None
trace = None

if RUN_LOCAL_PIPELINE:
    response = pipeline.answer(DEEP_DIVE_QUESTION)
    trace = response["question_trace"]
    print(f"Reranker load source: {response['reranker_load_source']}")
    display(Markdown("### Grounded answer\n\n" + response["answer"]))
else:
    display(Markdown("_Enable `RUN_LOCAL_PIPELINE` to execute the single-query walkthrough._"))

## 10. Candidate pool inspection

This table answers: *What did Phase 3 make eligible for reranking?* Candidate position is the post-RRF, cross-query order. Dense/BM25 provenance and raw scores remain diagnostic fields rather than inputs to an arithmetic average.

In [ ]:
if trace:
    candidate_frame = pd.DataFrame(trace["candidate_pool"])
    candidate_columns = [column for column in (
        "original_rrf_rank", "source", "page_number", "chunk_id", "score",
        "rrf_score", "retrieval_sources", "text_preview"
    ) if column in candidate_frame.columns]
    display(candidate_frame[candidate_columns])
else:
    candidate_frame = pd.DataFrame()

## 11. Reranker scoring inspection

Rank movement is more informative than a score in isolation. A large movement means the joint question/chunk model disagreed with the first-stage order. Thresholds are model-specific and must be qualified; they are not universal probabilities.

In [ ]:
if trace:
    reranked_frame = pd.DataFrame(trace["reranked_candidates"])
    reranked_frame["rank_movement"] = reranked_frame["original_rrf_rank"] - reranked_frame["reranked_rank"]
    display(reranked_frame[[
        "original_rrf_rank", "reranked_rank", "rank_movement",
        "reranker_score", "source", "page_number", "chunk_id"
    ]])
else:
    reranked_frame = pd.DataFrame()

## 12. Selected vs discarded evidence

This is the central Phase 4 decision surface. Below-threshold evidence may be retained through `adaptive_fallback` to protect answerability and is marked weak. Discarded chunks use only: `threshold_failed`, `redundancy`, `source_diversity_limit`, `token_budget`, `empty_text`, or `lower_rank_fallback`.

In [ ]:
if trace:
    selected_frame = pd.DataFrame(trace["selected_chunks"])
    discarded_frame = pd.DataFrame(trace["discarded_chunks"])
    display(Markdown("#### Selected"))
    display(selected_frame[[column for column in (
        "reranked_rank", "reranker_score", "source", "page_number",
        "chunk_id", "evidence_token_count", "selection_reason",
        "weak_evidence", "final_context_inclusion"
    ) if column in selected_frame.columns]])
    display(Markdown("#### Discarded"))
    display(discarded_frame[[column for column in (
        "reranked_rank", "reranker_score", "source", "chunk_id",
        "evidence_token_count", "discard_reason"
    ) if column in discarded_frame.columns]])
else:
    selected_frame, discarded_frame = pd.DataFrame(), pd.DataFrame()

## 13. Evidence quality scoring

Evidence quality is a transparent diagnostic, not a semantic correctness verdict. Per chunk it records reranker score, dense/BM25/both provenance, source-diversity contribution, citation availability, source/page/chunk metadata completeness, token count, and strong/medium/weak classification.

In [ ]:
if trace:
    quality_frame = pd.DataFrame(trace["evidence_quality"]["chunks"])
    display(quality_frame)
    display(pd.DataFrame([trace["evidence_quality"]["summary"]]))
else:
    quality_frame = pd.DataFrame()

## 14. Token reduction analysis

Candidate tokens represent the Phase 3-style serialized candidate context including citation headers. Selected evidence tokens measure retained chunk text. Final context tokens measure the exact prompt context after existing merging and token fitting. Reduction matters, but the diagnostics now flag reduction above 90%, candidate pools that produce zero selected chunks, answered questions below 500 selected tokens, and zero average selected score with non-empty candidates. These signals identify evidence starvation rather than celebrating it.

In [ ]:
if trace:
    token_frame = pd.DataFrame([
        ("Candidate context", trace["token_usage"]["candidate_tokens"]),
        ("Selected evidence", trace["token_usage"]["selected_evidence_tokens"]),
        ("Final context", trace["token_usage"]["final_context_tokens"]),
    ], columns=["stage", "tokens"])
    display(token_frame)
    print(f"Token reduction: {trace['token_usage']['token_reduction_percent']:.2f}%")
else:
    token_frame = pd.DataFrame()

## 15. Latency analysis

Phase 4 separates retrieval, reranking, selection, context construction, generation, and artifact export. Reranking adds latency; token reduction may reduce downstream generation cost. Both must be measured. A quality gain with unacceptable local latency is not a successful enterprise trade-off.

In [ ]:
if trace:
    latency_frame = pd.DataFrame([
        (key.removesuffix("_seconds").replace("_", " ").title(), value)
        for key, value in trace["latency"].items() if value is not None
    ], columns=["stage", "seconds"])
    display(latency_frame)
else:
    latency_frame = pd.DataFrame()

## 16. Citation quality inspection

Citations are built only from evidence that reaches final context. Source, page, and chunk completeness are inspected separately from whether a local PDF link can be constructed. An unsupported answer should return safe failure with no citations.

In [ ]:
if trace:
    citation_frame = pd.DataFrame(trace["citations"])
    display(citation_frame)
    for citation in trace["citations"]:
        if citation.get("pdf_link"):
            display(HTML(f'<a href="{citation["pdf_link"]}" target="_blank">Open {citation.get("source_file", "PDF")} at page {citation.get("page_number")}</a>'))
else:
    citation_frame = pd.DataFrame()

## 17. Decision visualizations

The following plots answer operational questions. They are intentionally simple, offline matplotlib views rather than decorative dashboards.

In [ ]:
def _plot_unavailable(title: str) -> None:
    print(f"{title}: run the single-query walkthrough first.")

if trace:
    print("Trace ready for decision visualizations.")
else:
    print("Visualization cells remain safe until a trace exists.")

### 17.1 Candidate pool funnel — where did chunks leave the pipeline?

In [ ]:
if trace:
    funnel = pd.Series({
        "Hybrid candidates": len(trace["candidate_pool"]),
        "Reranked": len(trace["reranked_candidates"]),
        "Selected": len(trace["selected_chunks"]),
        "Final context": len(trace["final_context_chunks"]),
    })
    funnel.plot.bar(title="Candidate pool funnel", color="#2563eb", ylabel="Chunks")
    plt.xticks(rotation=20); plt.tight_layout(); plt.show()
else: _plot_unavailable("Candidate pool funnel")

### 17.2 Reranker score distribution — is evidence clearly separated?

In [ ]:
if trace and not reranked_frame.empty:
    reranked_frame["reranker_score"].plot.hist(bins=min(10, len(reranked_frame)), color="#7c3aed", title="Reranker score distribution")
    plt.xlabel("Reranker score"); plt.tight_layout(); plt.show()
else: _plot_unavailable("Reranker score distribution")

### 17.3 Selected vs discarded — how aggressive was selection?

In [ ]:
if trace:
    pd.Series({"Selected": len(trace["selected_chunks"]), "Discarded": len(trace["discarded_chunks"])}).plot.bar(
        title="Selected vs discarded", color=["#15803d", "#be123c"], ylabel="Chunks"
    )
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else: _plot_unavailable("Selected vs discarded")

### 17.4 Token reduction — did selection reduce prompt context?

In [ ]:
if trace:
    token_frame.set_index("stage")["tokens"].plot.bar(title="Candidate to final context tokens", color="#0f766e", ylabel="Tokens")
    plt.xticks(rotation=20); plt.tight_layout(); plt.show()
else: _plot_unavailable("Token reduction")

### 17.5 Latency breakdown — where is wall-clock time spent?

In [ ]:
if trace and not latency_frame.empty:
    latency_frame.set_index("stage")["seconds"].plot.bar(title="Latency breakdown", color="#b45309", ylabel="Seconds")
    plt.xticks(rotation=30); plt.tight_layout(); plt.show()
else: _plot_unavailable("Latency breakdown")

### 17.6 Source diversity — did selection concentrate evidence?

In [ ]:
if trace:
    def source_set(items):
        return {str((item.get("metadata") or {}).get("source") or item.get("source")) for item in items}
    pd.Series({
        "Candidate sources": len(source_set(trace["candidate_pool"])),
        "Selected sources": len(source_set(trace["selected_chunks"])),
    }).plot.bar(title="Source diversity", color="#0369a1", ylabel="Unique documents")
    plt.xticks(rotation=15); plt.tight_layout(); plt.show()
else: _plot_unavailable("Source diversity")

### 17.7 Evidence strength — how much retained evidence is strong, medium, or weak?

In [ ]:
if trace:
    strength = pd.Series(trace["evidence_quality"]["summary"]["strength_distribution"])
    strength.plot.bar(title="Evidence strength distribution", color=["#15803d", "#d97706", "#be123c"], ylabel="Selected chunks")
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else: _plot_unavailable("Evidence strength")

### 17.8 Reranker rank movement — which candidates changed most?

In [ ]:
if trace and not reranked_frame.empty:
    movement = reranked_frame.set_index("chunk_id")[["original_rrf_rank", "reranked_rank"]]
    movement.plot.bar(title="Original RRF rank vs reranked rank", ylabel="Rank (lower is better)")
    plt.xticks(rotation=35); plt.tight_layout(); plt.show()
else: _plot_unavailable("Reranker rank movement")

### 17.9 Discard reason breakdown — why were chunks removed?

In [ ]:
if trace and not discarded_frame.empty:
    discarded_frame["discard_reason"].value_counts().plot.bar(title="Discard reason breakdown", color="#be123c", ylabel="Chunks")
    plt.xticks(rotation=25); plt.tight_layout(); plt.show()
else: _plot_unavailable("Discard reason breakdown")

### 17.10 Phase 3 vs Phase 4 comparison — what changed for this question?

In [ ]:
if trace and response:
    phase_comparison = pd.DataFrame([
        {
            "pipeline": "Phase 3 Hybrid candidate context",
            "answer_status": "not generated in this estimate",
            "context_tokens": trace["token_usage"]["candidate_tokens"],
            "citation_count": None,
            "selected_chunk_count": len(trace["candidate_pool"]),
        },
        {
            "pipeline": "Phase 4 Reranked Hybrid",
            "answer_status": response["answer_status"],
            "context_tokens": trace["token_usage"]["final_context_tokens"],
            "citation_count": len(trace["citations"]),
            "selected_chunk_count": len(trace["selected_chunks"]),
        },
    ])
    display(phase_comparison)
    print("This is a single-query engineering comparison, not benchmark qualification.")
else: _plot_unavailable("Phase 3 vs Phase 4 comparison")

## 18. Interactive manual QA

Edit `manual_questions` directly. Every executed question is exported. Set `INLINE_TRACE_LIMIT` to a positive integer for an intentional smoke cap; intentional large notebook runs use `export_only` and suppress inline trace rendering.

In [ ]:
manual_questions = [
    "What are the most important cybersecurity controls we should implement immediately?",
    "Rank the most critical cybersecurity recommendations from highest to lowest priority.",
    "Which recommendations should be implemented first for maximum risk reduction?",
    "Which controls provide the greatest improvement to cybersecurity maturity?",
    "What should be considered our top priorities before deploying new technology?",
    "Which vulnerabilities deserve immediate attention?",
    "Which governance controls are considered foundational?",
    "Which security controls should be implemented before advanced monitoring?",
    "What recommendations are considered mandatory versus recommended?",
    "How should cybersecurity investments be prioritized?",
    "Our board wants a one-page cybersecurity strategy. What should it contain?",
    "If we had only six months to improve cybersecurity, what should we focus on?",
    "Where should management invest first to reduce organizational cyber risk?",
    "What cybersecurity gaps are most likely to expose critical infrastructure?",
    "How can executive leadership measure cybersecurity maturity?",
    "How should executives track cybersecurity performance over time?",
    "Which cybersecurity metrics should appear in quarterly board meetings?",
    "What risks should executives never ignore?",
    "How should leadership balance security with operational continuity?",
    "What governance activities require executive oversight?", 
     "Show me every recommendation related to Zero Trust across all CERT-In publications.",
    "Find every mention of SBOM across the indexed documents.",
    "List every recommendation related to AI governance.",
    "Find all guidance related to incident response.",
    "Show every recommendation involving supply chain security.",
    "List every recommendation for continuous monitoring.",
    "Find every recommendation involving third-party vendors.",
    "Show all recommendations related to cloud security.",
    "List every recommendation related to vulnerability management.",
    "Find all references to privileged access management.",
 # unsupported behavior check
]
LARGE_RUN = True
INLINE_TRACE_LIMIT = config.max_inline_manual_questions

if INLINE_TRACE_LIMIT is not None and len(manual_questions) > INLINE_TRACE_LIMIT and not LARGE_RUN:
    print(f"Warning: using the first {INLINE_TRACE_LIMIT} questions. Set LARGE_RUN=True for export-only execution.")
    effective_manual_questions = manual_questions[:INLINE_TRACE_LIMIT]
else:
    effective_manual_questions = manual_questions

manual_run_mode = "export_only" if LARGE_RUN else "manual_qa"
config.allow_large_run = LARGE_RUN
print({"entered": len(manual_questions), "executing": len(effective_manual_questions), "mode": manual_run_mode})

In [ ]:
RUN_MANUAL_QA = True
manual_result = None

if RUN_MANUAL_QA:
    if not RUN_LOCAL_PIPELINE:
        raise RuntimeError("Set RUN_LOCAL_PIPELINE=True and initialize local indexes before manual QA.")
    manual_result = Phase4Runner(pipeline=pipeline, config=config).run(
        questions=effective_manual_questions,
        run_mode=manual_run_mode,
        run_metadata={"run_label": "notebook_manual_qa"},
    )
    print(f"Exported run: {manual_result.paths.root}")
else:
    print("Manual QA is gated. Set RUN_MANUAL_QA=True to answer and export.")

In [ ]:
if manual_result and not LARGE_RUN:
    manual_rows = pd.read_csv(manual_result.paths.results_csv, encoding="utf-8-sig")
    manual_traces = json.loads(manual_result.paths.retrieval_json.read_text(encoding="utf-8"))
    display(manual_rows[[
        "question", "answer_status", "evidence_confidence", "fallback_used",
        "selected_chunk_count", "discarded_chunk_count", "selected_evidence_tokens",
        "final_context_tokens", "token_reduction_percent", "citation_count",
        "total_latency_seconds"
    ]])
    for item in manual_traces:
        display(Markdown(f"### {item['question']}\n\n{item['answer']}"))
        display(pd.DataFrame(item["evidence_quality"]["chunks"]))
        display(pd.DataFrame(item["selected_chunks"]))
        display(pd.DataFrame(item["discarded_chunks"]))
elif manual_result:
    print("Large run completed in export-only mode; inspect the artifact bundle instead of rendering all traces inline.")

## 19. Smoke comparison

A smoke run checks execution and artifact integrity on a small question set. It can reveal obvious score, selection, citation, or latency failures, but it cannot qualify Phase 4. Use the same corpus, generation model, question, and token settings for a Phase 3/Phase 4 comparison.

In [ ]:
RUN_SMOKE_COMPARISON = False
smoke_questions = effective_manual_questions[:3]

if RUN_SMOKE_COMPARISON:
    if not RUN_LOCAL_PIPELINE:
        raise RuntimeError("Initialize local Phase 4 resources before the smoke run.")
    smoke_result = Phase4Runner(pipeline=pipeline, config=config).run(
        questions=smoke_questions,
        run_mode="smoke",
        run_metadata={"comparison_scope": "execution_smoke"},
    )
    display(pd.DataFrame([smoke_result.metrics]))
else:
    print("Smoke comparison skipped.")

## 20. Optional full benchmark qualification

The full benchmark is deliberately gated because local generation and cross-encoder scoring are expensive. Qualification requires separate Phase 3 Hybrid and Phase 4 Reranked Hybrid bundles over the unchanged frozen benchmark, followed by comparison of answer quality, citation quality, unsupported-question behavior, context tokens, token reduction, latency, selected/discarded chunks, average reranker score, and evidence-strength distribution.

In [ ]:
RUN_FULL_BENCHMARK = False

if RUN_FULL_BENCHMARK:
    if not RUN_LOCAL_PIPELINE:
        raise RuntimeError("Initialize the local pipeline before qualification.")
    frozen_benchmark = load_benchmark(
        config.benchmark_csv_path,
        metadata_path=config.benchmark_metadata_path,
    )
    phase4_qualification = Phase4Runner(pipeline=pipeline, config=config).run(
        benchmark=frozen_benchmark,
        run_mode="benchmark",
        run_metadata={"qualification": "pending_review"},
    )
    print(phase4_qualification.paths.root)
    print("Run the frozen Phase 3 Hybrid configuration separately and compare both bundles before changing qualification status.")
else:
    print("Full benchmark skipped. Phase 4 remains implemented but not benchmark-qualified.")

## 21. Export artifact inspection

Every Phase 4 runner execution writes `results.csv`, `results.xlsx`, `report.html`, `config.json`, `summary.json`, `metrics.json`, `retrieval.json`, `logs.txt`, `context/`, and decision SVGs under `figures/`. CSV remains machine-readable; XLSX provides clickable citation links; HTML is standalone and embeds its visualizations without CDNs.

In [ ]:
artifact_result = manual_result if manual_result is not None else None
if artifact_result:
    for artifact in sorted(artifact_result.paths.root.rglob("*")):
        print(artifact.relative_to(artifact_result.paths.root))
    required = [
        artifact_result.paths.results_csv, artifact_result.paths.results_xlsx,
        artifact_result.paths.report_html, artifact_result.paths.config_json,
        artifact_result.paths.summary_json, artifact_result.paths.metrics_json,
        artifact_result.paths.retrieval_json, artifact_result.paths.logs,
    ]
    assert all(path.is_file() for path in required)
else:
    expected = config.output_root / config.phase_output_name / f"{config.run_prefix}_<timestamp>"
    print(f"Configured output pattern: {expected}")

## 22. Advantages

- Separates recall-oriented retrieval from precision-oriented evidence decisions.
- Keeps dense, BM25, RRF, and cross-encoder scores interpretable instead of averaging incompatible scales.
- Records why every chunk is retained or discarded.
- Reduces context intentionally before generation rather than only truncating at the model limit.
- Uses dependency injection and a mock reranker for fast automated tests.
- Preserves Phase 3 retrieval, token, citation, evaluation, and artifact contracts.
- Produces decision-focused offline notebook and HTML diagnostics.

## 23. Limitations

- Full frozen-benchmark qualification has not been run; score thresholds and quality gains are not yet benchmark-qualified.
- Cross-encoder scores are model-specific and should not be interpreted as universal probabilities.
- Lexical Jaccard redundancy is deterministic but not semantic contradiction analysis.
- Source diversity can reduce concentration, but multiple sources do not prove correctness or independence.
- Local reranking adds model memory and latency.
- OCR/image-only content depends on the existing loaders; Phase 4 does not add visual understanding.

## 24. Enterprise considerations

On-premise deployment should stage and approve reranker weights, record model/version hashes, select CPU/GPU and batch settings from measured capacity, monitor weak-evidence and single-source diagnostics, retain full traces for audited workflows, and use compact traces for high-volume operation. Threshold changes are configuration changes that require benchmark comparison and review. Retrieval-time authorization remains a future production control and must be enforced before this pipeline is exposed to multiple permission domains.

## 25. Automated testing readiness

The reusable interfaces allow tests to call the reranker, selector, quality scorer, trace serializer, reporting layer, and complete pipeline independently. `MockReranker` makes relevance scores deterministic. Artifact tests do not load the real model or call Ollama. Large future benchmark runners can select compact traces and export-only mode without executing this notebook.

## 26. Phase 4.5 preview — deferred scope

Phase 4.5 is reserved for research and design work. The following are **deferred and not implemented in Phase 4**:

- visual document understanding;
- multimodal retrieval; and
- contradiction detection.

Lexical redundancy removal is not contradiction detection, and PDF links are not visual document understanding.

## 27. Preparing for Phase 5 Agentic RAG

Phase 5 can consume Phase 4's selected evidence and explicit weak-evidence diagnostics when deciding whether a question needs decomposition or another bounded retrieval step. Agentic behavior should not compensate for an unqualified retrieval baseline. Planning, repeated retrieval, verification, and stop conditions remain Phase 5 work.

## 28. Conclusion

Phase 4 adds a measurable precision-control layer between hybrid retrieval and generation. Its engineering claim is implementation completeness and testability: local reranking, explainable evidence selection, evidence-quality diagnostics, token reduction accounting, trace serialization, and compatible artifacts are present. Its empirical answer-quality claim remains open until the gated Phase 3 versus Phase 4 benchmark is completed and reviewed.

In [ ]:
if getattr(pipeline, "client", None) is not None:
    pipeline.close()
    print("Local resources closed.")

## Enterprise File Format Readiness

Phase 4 now validates the enterprise corpus through one backend file-format registry before extraction, chunking, embedding, indexing, and reporting.

- `SUPPORTED_NOW`: fully processable by the current ingestion pipeline.
- `OCR_SUPPORTED`: image formats processed through OCR before normal chunking and indexing.
- `RECOGNIZED_FUTURE_SUPPORT`: known enterprise formats that are intentionally skipped until parsers are implemented.
- `UNSUPPORTED`: unknown formats that are rejected and reported.

Supported documents are ingested now. OCR image formats require OCR extraction before chunking. Future-support formats are recognized for planning and settings UX, but they are not silently ingested. Unsupported files are rejected and flagged.


In [ ]:
import pandas as pd
from cial_knowledge_os import (
    list_formats_by_category,
    scan_file_format_readiness,
)

registry_rows = []
for category, payload in list_formats_by_category().items():
    for item in payload["formats"]:
        registry_rows.append({
            "category": category,
            "format_label": item["format_label"],
            "extensions": ", ".join(item["extensions"]),
            "support_status": item["support_status"],
            "ingestion_enabled": item["ingestion_enabled"],
            "requires_ocr": item["requires_ocr"],
            "notes": item["backend_notes"],
        })

format_registry_df = pd.DataFrame(registry_rows)
format_registry_df


In [ ]:
file_format_readiness = scan_file_format_readiness(config.knowledge_root)
readiness_summary = pd.DataFrame([
    {"metric": "total_files", "value": file_format_readiness["total_files"]},
    {"metric": "processable_files", "value": file_format_readiness["processable_files"]},
    {"metric": "ocr_files", "value": file_format_readiness["ocr_files"]},
    {"metric": "recognized_future_files", "value": file_format_readiness["recognized_future_files"]},
    {"metric": "unsupported_files", "value": file_format_readiness["unsupported_files"]},
])
extension_distribution_df = pd.DataFrame(file_format_readiness["extensions"])
support_status_df = pd.DataFrame(
    file_format_readiness["support_status_distribution"].items(),
    columns=["support_status", "count"],
)
category_distribution_df = pd.DataFrame(
    file_format_readiness["category_distribution"].items(),
    columns=["category", "count"],
)
readiness_summary


In [ ]:
display(extension_distribution_df)
display(support_status_df)
display(category_distribution_df)

if not extension_distribution_df.empty:
    extension_distribution_df.head(20).plot.bar(
        x="extension", y="count", title="File Extension Distribution"
    )
if not support_status_df.empty:
    support_status_df.plot.bar(
        x="support_status", y="count", title="Support Status Distribution"
    )
if not category_distribution_df.empty:
    category_distribution_df.plot.bar(
        x="category", y="count", title="Category Distribution"
    )


### OCR Processing Summary

OCR-supported PNG, JPG/JPEG, and TIFF files are validated as images, preprocessed, extracted through the configured OCR engine, cleaned, and then passed to the same chunking, embedding, and indexing path used by other extracted text.

The default OCR engine is Tesseract through `pytesseract` and Pillow. Current settings are `ocr_enabled`, `ocr_engine`, `ocr_preprocessing`, and `ocr_language` on the active Phase 4 configuration. Future OCR engines such as EasyOCR, PaddleOCR, Google Vision, Azure Document Intelligence, or AWS Textract can be enabled by adding a backend behind the OCR interface and changing configuration.


In [ ]:
ocr_settings = {
    "ocr_enabled": config.ocr_enabled,
    "ocr_engine": config.ocr_engine,
    "ocr_preprocessing": config.ocr_preprocessing,
    "ocr_language": config.ocr_language,
    "supported_image_formats": "png, jpg, jpeg, tiff, tif",
    "pipeline": "Image validation -> preprocessing -> OCR extraction -> text cleaning -> chunking -> embedding -> vector store",
}
pd.DataFrame(ocr_settings.items(), columns=["setting", "value"])


### Engineering Notes

- Enabling a future parser requires changing the central registry status and adding an extraction handler.
- Registry data feeds ingestion validation, OCR routing, chunking eligibility, embedding/indexing eligibility, Phase 4 reports, dataset scanning, and future frontend settings.
- Recognized future formats are skipped with warnings so corpus readiness can be planned without accidental ingestion.
- Unsupported formats are rejected and included in readiness diagnostics.
